# inferencing using descbert

In [ ]:
#!pip install evaluate
#!pip install torchao --upgrade -q

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from nltk.tokenize import sent_tokenize, word_tokenize
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, DataCollatorWithPadding, TrainingArguments, Trainer
from datasets import load_dataset, DatasetDict, Dataset
from tqdm import tqdm
from evaluate import evaluator
from sklearn.model_selection import train_test_split
import time

C:\Users\grete\miniconda3\envs\env_nltk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
#use PEFT
from peft import PeftModel, PeftConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

model_name = "e0572777/descbert"

# Load the PEFT config
peft_config = PeftConfig.from_pretrained(model_name)

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path,
    num_labels=2
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)

# Load the LoRA adapter 
model = PeftModel.from_pretrained(base_model, model_name)
model = model.merge_and_unload()  # merges LoRA weights into the base model

model.eval() 

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, truncation=True, max_length=512, batch_size=32)

C:\Users\grete\miniconda3\envs\env_nltk\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\grete\.cache\huggingface\hub\models--e0572777--descbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1189.11it/s]
[transformers] BertForSequenceClassificati

In [5]:
#batch titles for inferencing with folder of 430 plays
sentence_df = pd.read_csv("dataset_full.csv", encoding="utf-8-sig")
sentence_df = sentence_df.iloc[0:100]

# Drop NaN rows and reset index
sentence_df = sentence_df.dropna(subset=["sentence_gpt4"]).reset_index(drop=True)

# to string
sentence_df["sentence_gpt4"] = sentence_df["sentence_gpt4"].astype(str)

# Print check for any remaining non-strings
non_strings = [(i, type(v)) for i, v in enumerate(sentence_df["sentence_gpt4"]) if not isinstance(v, str)]
if non_strings:
    print(f"WARNING: {len(non_strings)} non-string values found: {non_strings[:5]}")
else:
    print(f"All {len(sentence_df)} values are strings")

sentences = sentence_df["sentence_gpt4"].tolist()
filename = sentence_df["filename"].tolist()

C:\Users\grete\AppData\Local\Temp\ipykernel_19264\2822260073.py:2: DtypeWarning: Columns (0: note) have mixed types. Specify dtype option on import or set low_memory=False.
  sentence_df = pd.read_csv("dataset_full.csv", encoding="utf-8-sig")


All 100 values are strings


In [6]:
from datasets import Dataset as HFDataset

# Convert DataFrame column directly to HF Dataset
hf_dataset = HFDataset.from_pandas(sentence_df[["filename", "sentence_number", "sentence_gpt4"]])

In [7]:
#inferencing with HF datasets

raw_preds = classifier(
    list(hf_dataset["sentence_gpt4"]),
    batch_size=32,
    num_workers=0
)
#sort results into dataframe
results_df = sentence_df.copy()
results_df["label"] = None
results_df["score"] = None

for i, pred in enumerate(raw_preds):
    results_df.at[i, "label"] = pred["label"]
    results_df.at[i, "score"] = pred["score"]

results_df.to_csv("predicted_labels.csv", index=False, encoding="utf-8-sig")
print(results_df.head())

           filename play_title      speaker_id speaker_name speaker_spblock  \
0  alexander-caesar     Caesar  eng000261-juno         Juno           Iuno.   
1  alexander-caesar     Caesar  eng000261-juno         Juno           Iuno.   
2  alexander-caesar     Caesar  eng000261-juno         Juno           Iuno.   
3  alexander-caesar     Caesar  eng000261-juno         Juno           Iuno.   
4  alexander-caesar     Caesar  eng000261-juno         Juno           Iuno.   

   gender  sentence_number                                  sentence_original  \
0  FEMALE                1  THough I a goddesse glance through th'azure ro...   
1  FEMALE                2  And though I banquet in th'Aetheriall bowres, ...   
2  FEMALE                3  In vaine, vaine mortals seeke for helpe at me,...   
3  FEMALE                4    Might Pallas once drowne thousands in the seas.   
4  FEMALE                5                   And metamorphose Diomedes mates?   

                                      